# Uji penerimaan SLR_09 v0.8.2

Acuan FINAL082 telah dikembalikan pengguna dengan 50 APPROVED dan 671 token. Salinan review dan snapshot seluruh sumber terikat disimpan sebelum model dimuat. Notebook memvalidasi ulang, membekukan acuan, lalu menjalankan inferensi CPU dengan bobot SENA/NARA dan target yang tetap. Hasil eksekusi sebenarnya dicatat pada salinan _executed; penerimaan mutu tidak otomatis.


In [1]:
from pathlib import Path
import os,sys,json,subprocess
from datetime import datetime,timezone
ROOT=Path(os.environ['SLR09_PACKAGE_ROOT']).resolve()
WORK=Path(os.environ['SLR09_WORK_DIR']).resolve()
REVIEW=Path(os.environ['SLR09_REVIEW_XLSX']).resolve()
SENA=Path(os.environ['SLR09_SENA_ARCHIVE']).resolve()
NARA=Path(os.environ['SLR09_NARA_ARCHIVE']).resolve()
sys.path.insert(0,str(ROOT/'code/acceptance'))
from import_review_xlsx import convert
from evaluate_acceptance import validate,write


In [2]:
# Penjagaan ini berjalan sebelum model dimuat atau acuan hasil impor disimpan.
bundle=convert(REVIEW,root=ROOT,validate_ready=True)
rows,preflight=validate(bundle,ROOT)
RUN=WORK/('final_acceptance_v082_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
RUN.mkdir(parents=True,exist_ok=False)
reviewed=RUN/'reviewed_reference.json'
write(reviewed,bundle)
print(json.dumps(preflight,ensure_ascii=False,indent=2))


{
  "records": 50,
  "class_support": {
    "FLOOD": 10,
    "EARTHQUAKE": 10,
    "FIRE": 10,
    "ACCIDENT": 10,
    "NON_EVENT": 10
  },
  "role_support": {
    "OFFICER-ARG": 32,
    "O": 244,
    "FLOOD-EVENT": 10,
    "INFORMATION-ARG": 43,
    "AFFECTEDOBJECTS-ARG": 52,
    "PLACE-ARG": 84,
    "TIME-ARG": 111,
    "REASON-ARG": 12,
    "WOUNDVICTIM-ARG": 15,
    "STREET-ARG": 14,
    "DEATHVICTIM-ARG": 12,
    "EARTHQUAKE-EVENT": 11,
    "FIRE-EVENT": 10,
    "ACCIDENT-EVENT": 10,
    "FALSE-EVENT": 11
  },
  "sources": {
    "ASSISTANT_AUTHORED_SYNTHETIC": 50
  },
  "excluded": [],
  "known_data_overlap": "NO_NORMALIZED_EXACT_MATCH_IN_AVAILABLE_SOURCES",
  "historical_initial_training_membership_verified": false,
  "population_generalization_established": false
}


In [3]:
cmd=[sys.executable,str(ROOT/'code/acceptance/evaluate_acceptance.py'),'--root',str(ROOT),'--review',str(reviewed),'--output',str(RUN/'evidence'),'--sena-zip',str(SENA),'--nara-zip',str(NARA),'--device','cpu','--environment-label','CODEX_CPU_REPRESENTATIVE']
process=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in process.stdout:print(line,end='',flush=True)
process.wait()
if process.returncode:raise RuntimeError('Evaluasi gagal; periksa workflow_result dan error_trace yang disimpan.')
report=json.loads((RUN/'evidence/evaluation.json').read_text())
print(json.dumps({k:report[k] for k in ['checks','metric_targets_met','decision','quality_acceptance_approved','card_done']},ensure_ascii=False,indent=2))


Evaluasi 1 / 50
Evaluasi 10 / 50
Evaluasi 20 / 50
Evaluasi 30 / 50
Evaluasi 40 / 50
Evaluasi 50 / 50
{
  "schema": "slr09-acceptance-workflow-0.4",
  "status": "SUCCEEDED",
  "phase": "COMPLETE",
  "started_at": "2026-10-19T12:45:30.232041+00:00",
  "model_inference_executed": true,
  "training_executed": false,
  "quality_acceptance_approved": false,
  "card_done": false,
  "target_vm_verified": false,
  "environment_scope": "REPRESENTATIVE",
  "environment_label": "CODEX_CPU_REPRESENTATIVE",
  "frozen_bundle_sha256": "bf4a23100c37dcfaaccb878fb58d3c4fd0c28bf05e27f98598bbc4b8392482dd",
  "models": {
    "sena": {
      "model_id": "SILAR_SENA_context_guard_v081_20261019T104620",
      "run_id": "20261019T104620",
      "archive_sha256_expected": "760a79f6951fc8c380aeed6779006444e3b1992ae7d720af9d80f2be4c24a20d",
      "files": {
        "config.json": "febb4294e220615be653133104562be26236ea97165c5103dc7b6d377277da9a",
        "model.safetensors": "d7a8035b2db81d5ad2d166162fb0633327ce7c

Jika gagal dan hasil dipakai untuk perbaikan, seluruh acuan yang telah diuji menjadi development. Siapkan acuan baru untuk putaran penerimaan berikutnya. Jangan mengubah target atau label setelah melihat prediksi untuk membuat putaran ini lulus.
